In [0]:
from pyspark.sql import functions as F

In [0]:
df = spark.table("dev_l2.mia.customer_feedback")


**Loading Data**

In [0]:
display(df)

Customer,Customer Feedback
Rushab,"Great support and fast resolution"","
Tejasini,Excellent service and quick response
Subodh,Very satisfied with the support team
Rahul,Issue resolved quickly and efficiently
Vaibhavi,Outstanding experience and communication
Ramesh,Very slow response and poor service
Raju,Bad experience and delayed support
Chanda,Average service experience
Jeevan,Neither good nor bad experience


In [0]:
df.printSchema()

root
 |-- Customer: string (nullable = true)
 |-- Customer Feedback: string (nullable = true)



In [0]:
print(df.columns)

['Customer', 'Customer Feedback']


In [0]:
from pyspark.sql.functions import col
df.filter(col("Customer Feedback").isNull()).count()

0

**Cleaning Data**

In [0]:
from pyspark.sql.functions import col, trim
df_clean = (
    df
    .filter(col("Customer Feedback").isNotNull())
    .withColumn("Customer Feedback", trim(col("Customer Feedback")))
    .filter(col("Customer Feedback") != "")
)
display(df_clean)

Customer,Customer Feedback
Rushab,"Great support and fast resolution"","
Tejasini,Excellent service and quick response
Subodh,Very satisfied with the support team
Rahul,Issue resolved quickly and efficiently
Vaibhavi,Outstanding experience and communication
Ramesh,Very slow response and poor service
Raju,Bad experience and delayed support
Chanda,Average service experience
Jeevan,Neither good nor bad experience


**Sentiment Analysis**

In [0]:
%pip install transformers torch

Note: you may need to restart the kernel using dbutils.library.restartPython() to use updated packages.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 68.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.6/526.6 MB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 770.3/770.3 kB 45.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 793.5/793.5 kB 45.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 784.9/784.9 kB 47.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.9/122.9 kB 8.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 84.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 7.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 37.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 818.2/818.2 kB 31.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 81.2 MB/s eta 0:00:00
     ━━━━━

In [0]:
dbutils.library.restartPython()

In [0]:
from pyspark.sql import functions as F

df = spark.table("dev_l2.mia.customer_feedback")

display(df)

Customer,Customer Feedback
Rushab,"Great support and fast resolution"","
Tejasini,Excellent service and quick response
Subodh,Very satisfied with the support team
Rahul,Issue resolved quickly and efficiently
Vaibhavi,Outstanding experience and communication
Ramesh,Very slow response and poor service
Raju,Bad experience and delayed support
Chanda,Average service experience
Jeevan,Neither good nor bad experience


In [0]:
import transformers
import torch

print("Transformers:", transformers.__version__)
print("Torch:", torch.__version__)

Transformers: 5.15.0
Torch: 2.13.0+cu130


In [0]:
from transformers import pipeline

sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

In [0]:
result = sentiment_pipeline("I absolutely love this product!")

print(result)

[{'label': 'POSITIVE', 'score': 0.9998854398727417}]


In [0]:
def analyze_sentiment(text):
    result = sentiment_pipeline(text)[0]
    
    return (
        result["label"],
        float(result["score"])
    )

In [0]:
print(analyze_sentiment("The product is excellent!"))

('POSITIVE', 0.9998801946640015)


In [0]:
sample_df = df.limit(10)
display(sample_df) 

Customer,Customer Feedback
Rushab,"Great support and fast resolution"","
Tejasini,Excellent service and quick response
Subodh,Very satisfied with the support team
Rahul,Issue resolved quickly and efficiently
Vaibhavi,Outstanding experience and communication
Ramesh,Very slow response and poor service
Raju,Bad experience and delayed support
Chanda,Average service experience
Jeevan,Neither good nor bad experience


In [0]:
pdf = sample_df.toPandas()
display(pdf)

Customer,Customer Feedback
Rushab,"Great support and fast resolution"","
Tejasini,Excellent service and quick response
Subodh,Very satisfied with the support team
Rahul,Issue resolved quickly and efficiently
Vaibhavi,Outstanding experience and communication
Ramesh,Very slow response and poor service
Raju,Bad experience and delayed support
Chanda,Average service experience
Jeevan,Neither good nor bad experience


In [0]:
sentiment_results = []

for text in pdf["Customer Feedback"]:
    label, score = analyze_sentiment(text)
    
    sentiment_results.append({
        "Sentiment": label,
        "SentimentScore": score
    })

In [0]:
import pandas as pd

sentiment_results_df = pd.DataFrame(sentiment_results)

pdf["Sentiment"] = sentiment_results_df["Sentiment"]
pdf["SentimentScore"] = sentiment_results_df["SentimentScore"]

In [0]:
display(pdf)

Customer,Customer Feedback,Sentiment,SentimentScore
Rushab,"Great support and fast resolution"",",POSITIVE,0.9998759031295776
Tejasini,Excellent service and quick response,POSITIVE,0.9998533725738525
Subodh,Very satisfied with the support team,POSITIVE,0.9996887445449829
Rahul,Issue resolved quickly and efficiently,POSITIVE,0.9996084570884705
Vaibhavi,Outstanding experience and communication,POSITIVE,0.9998729228973389
Ramesh,Very slow response and poor service,NEGATIVE,0.9997891783714294
Raju,Bad experience and delayed support,NEGATIVE,0.9997840523719788
Chanda,Average service experience,NEGATIVE,0.9464848041534424
Jeevan,Neither good nor bad experience,NEGATIVE,0.9992305040359497


In [0]:
sentiment_df = spark.createDataFrame(pdf)

display(sentiment_df)

Customer,Customer Feedback,Sentiment,SentimentScore
Rushab,"Great support and fast resolution"",",POSITIVE,0.9998759031295776
Tejasini,Excellent service and quick response,POSITIVE,0.9998533725738525
Subodh,Very satisfied with the support team,POSITIVE,0.9996887445449829
Rahul,Issue resolved quickly and efficiently,POSITIVE,0.9996084570884705
Vaibhavi,Outstanding experience and communication,POSITIVE,0.9998729228973389
Ramesh,Very slow response and poor service,NEGATIVE,0.9997891783714294
Raju,Bad experience and delayed support,NEGATIVE,0.9997840523719788
Chanda,Average service experience,NEGATIVE,0.9464848041534424
Jeevan,Neither good nor bad experience,NEGATIVE,0.9992305040359497


In [0]:
sentiment_summary = (
    sentiment_df
    .groupBy("Sentiment")
    .count()
    .orderBy(F.desc("count"))
)

display(sentiment_summary)

Sentiment,count
POSITIVE,5
NEGATIVE,4


In [0]:
total = sentiment_df.count()

sentiment_percentage = (
    sentiment_df
    .groupBy("Sentiment")
    .count()
    .withColumn(
        "Percentage",
        F.round(F.col("count") * 100 / total, 2)
    )
    .orderBy(F.desc("count"))
)

display(sentiment_percentage)

Sentiment,count,Percentage
POSITIVE,5,55.56
NEGATIVE,4,44.44


In [0]:
negative_feedback = (
    sentiment_df
    .filter(F.col("Sentiment") == "NEGATIVE")
    .select(
        "Customer",
        "Customer Feedback",
        "Sentiment",
        "SentimentScore"
    )
)

display(negative_feedback)

Customer,Customer Feedback,Sentiment,SentimentScore
Ramesh,Very slow response and poor service,NEGATIVE,0.9997891783714294
Raju,Bad experience and delayed support,NEGATIVE,0.9997840523719788
Chanda,Average service experience,NEGATIVE,0.9464848041534424
Jeevan,Neither good nor bad experience,NEGATIVE,0.9992305040359497
